<a href="https://colab.research.google.com/github/HEM2058/remotesening-app/blob/main/lidar_las.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import laspy

# Step 1: Load the .las file
las_file = '/content/bhoj.las'
las = laspy.read(las_file)

# Step 2: Print metadata
print("=== LAS File Metadata ===")
print(f"Version: {las.header.version}")
print(f"Point Format ID: {las.header.point_format.id}")
print(f"Number of Points: {las.header.point_count}")
print(f"Point Format Dimensions: {las.point_format.dimension_names}")
print(f"Scale Factors: {las.header.scales}")
print(f"Offsets: {las.header.offsets}")
print(f"X Range: {las.header.min[0]} - {las.header.max[0]}")
print(f"Y Range: {las.header.min[1]} - {las.header.max[1]}")
print(f"Z Range: {las.header.min[2]} - {las.header.max[2]}")


=== LAS File Metadata ===
Version: 1.2
Point Format ID: 3
Number of Points: 49402345
Point Format Dimensions: <generator object PointFormat.dimension_names.<locals>.<genexpr> at 0x789bd67dacf0>
Scale Factors: [0.001 0.001 0.001]
Offsets: [ 420310.30845004 3093204.46661418       0.        ]
X Range: 420492.3304500405 - 422001.98045004054
Y Range: 3093282.02961418 - 3094482.6196141797
Z Range: 219.625 - 4092.031


In [ ]:
import laspy
import numpy as np
import folium
from pyproj import Transformer

# Load the LAS file
las = laspy.read('/content/bhoj.las')

# Get X, Y, Z coordinates
x = las.x
y = las.y
z = las.z

# Downsample for visualization (optional: keeps map responsive)
sample_size = 50000
indices = np.random.choice(len(x), sample_size, replace=False)
x_sample = x[indices]
y_sample = y[indices]
z_sample = z[indices]

# Convert UTM (EPSG:32645) to lat/lon (EPSG:4326) — change if your CRS is different
transformer = Transformer.from_crs("epsg:32645", "epsg:4326", always_xy=True)
lon, lat = transformer.transform(x_sample, y_sample)

# Create folium map centered on average coordinates
map_center = [np.mean(lat), np.mean(lon)]
m = folium.Map(location=map_center, zoom_start=14, tiles='OpenStreetMap')

# Add each point to the map as a tiny circle marker
for i in range(len(lat)):
    folium.CircleMarker(
        location=[lat[i], lon[i]],
        radius=1,
        color='blue',
        fill=True,
        fill_opacity=0.5
    ).add_to(m)

# Display the map inline
m


In [ ]:
import laspy
import numpy as np
import folium
from pyproj import Transformer

# Load the LAS file
las = laspy.read('/content/bhoj.las')

# Get X, Y, Z coordinates
x = las.x
y = las.y
z = las.z

# Downsample for visualization (optional: keeps map responsive)
sample_size = 50000
indices = np.random.choice(len(x), sample_size, replace=False)
x_sample = x[indices]
y_sample = y[indices]
z_sample = z[indices]

# Convert UTM (EPSG:32645) to lat/lon (EPSG:4326)
transformer = Transformer.from_crs("epsg:32645", "epsg:4326", always_xy=True)
lon, lat = transformer.transform(x_sample, y_sample)

# Create folium map centered on average coordinates, no default tiles
map_center = [np.mean(lat), np.mean(lon)]
m = folium.Map(location=map_center, zoom_start=14, tiles=None)

# Add Google Satellite layer
folium.TileLayer(
    tiles='http://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite',
    overlay=False,
    control=True
).add_to(m)

# Add each point to the map as a tiny circle marker
for i in range(len(lat)):
    folium.CircleMarker(
        location=[lat[i], lon[i]],
        radius=1,
        color='blue',
        fill=True,
        fill_opacity=0.5
    ).add_to(m)

# Display the map
m


In [14]:
import laspy
import numpy as np

# Load the LAS file
las = laspy.read('/content/bhoj.las')

# Check and print all unique classifications
classes, counts = np.unique(las.classification, return_counts=True)
print("=== Point Classifications in Dataset ===")
for cls, count in zip(classes, counts):
    print(f"Class {cls}: {count} points")


=== Point Classifications in Dataset ===
Class 2: 4254081 points
Class 5: 45148264 points


In [ ]:
import laspy
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# Load LAS file
las = laspy.read('/content/bhoj.las')

# Extract coordinates and classification
x, y, z = las.x, las.y, las.z
classification = las.classification

# Filter ground (class 2) and high vegetation (class 5)
ground_mask = classification == 2
veg_mask = classification == 5

x_ground, y_ground, z_ground = x[ground_mask], y[ground_mask], z[ground_mask]
x_veg, y_veg, z_veg = x[veg_mask], y[veg_mask], z[veg_mask]

# Define grid resolution (in same units as x/y, typically meters)
grid_res = 2  # 2 meters
x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()
grid_x, grid_y = np.mgrid[x_min:x_max:grid_res, y_min:y_max:grid_res]

# Interpolate: generate gridded surfaces
dtm = griddata((x_ground, y_ground), z_ground, (grid_x, grid_y), method='linear')
dsm = griddata((x_veg, y_veg), z_veg, (grid_x, grid_y), method='linear')

# Canopy Height Model (CHM) = DSM - DTM
chm = dsm - dtm

# Plot DTM, DSM, CHM
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
titles = ['DTM (Ground)', 'DSM (Vegetation)', 'CHM (Canopy Height)']
datasets = [dtm, dsm, chm]

for ax, data, title in zip(axs, datasets, titles):
    im = ax.imshow(data.T, cmap='terrain', origin='lower',
                   extent=(x_min, x_max, y_min, y_max))
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.7)

plt.tight_layout()
plt.show()


In [ ]:
import laspy
import numpy as np
import folium
from pyproj import Transformer

# Load the LAS file
las = laspy.read('/content/bhoj.las')

# Get X, Y, Z coordinates
x = las.x
y = las.y
z = las.z

# Downsample for visualization (optional: keeps map responsive)
sample_size = 50000
indices = np.random.choice(len(x), sample_size, replace=False)
x_sample = x[indices]
y_sample = y[indices]
z_sample = z[indices]

# Convert UTM (EPSG:32645) to lat/lon (EPSG:4326) — change if your CRS is different
transformer = Transformer.from_crs("epsg:32645", "epsg:4326", always_xy=True)
lon, lat = transformer.transform(x_sample, y_sample)

# Create folium map centered on average coordinates
map_center = [np.mean(lat), np.mean(lon)]
m = folium.Map(location=map_center, zoom_start=14, tiles='OpenStreetMap')

# Add each point to the map as a tiny circle marker
for i in range(len(lat)):
    folium.CircleMarker(
        location=[lat[i], lon[i]],
        radius=1,
        color='blue',
        fill=True,
        fill_opacity=0.5
    ).add_to(m)

# Display the map inline
m


In [11]:
pip install geemap


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.3 MB/s eta 0:00:00


In [2]:
pip install laspy numpy matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.1 MB/s eta 0:00:00
